# Experiment: exp_002 train_audio reproduction

Objective:
- Train a repository-native baseline on isolated `train_audio` using the same core architecture as the current reference checkpoints.
- Measure how much of the current public baseline can be recovered without external checkpoints.
- Produce a clean starting point for later soundscape finetuning.

Success criteria:
- Build a reproducible train/valid split over isolated recordings.
- Train EfficientNet-B0 + GeM + attention SED head with the reference mel frontend.
- Save history and the best checkpoint under `experiments/outputs/exp_002_train_audio_reproduction/`.


In [1]:
import sys
print(sys.executable)

/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/bin/python


In [2]:
from __future__ import annotations

import json
import math
import random
import sys
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from birdclef2026.reference_eval import (
    ReferenceModelConfig,
    build_reference_model_components,
    score_macro_auc,
)

PROJECT_ROOT


PosixPath('/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026')

## Plan

- Hypothesis: a clean reproduction of the reference architecture on isolated `train_audio` should give us a strong native baseline, even before soundscape finetuning.
- Controlled change: no soundscape data yet, no checkpoint borrowing, and no aggressive augmentations in the first pass.
- Primary metrics:
  - hold-out macro ROC-AUC on isolated recordings
  - train/valid loss curves
  - number of classes actually scored on the validation split


In [ ]:
@dataclass
class TrainConfig:
    seed: int = 42
    sample_rate: int = 32_000
    clip_duration: float = 5.0
    valid_frac: float = 0.15
    batch_size: int = 24
    num_workers: int = 0
    epochs: int = 8
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    use_amp: bool = True
    use_secondary_labels: bool = False
    max_rows: int | None = None
    max_samples_per_class: int | None = None
    save_every_epoch: bool = False

    @property
    def clip_samples(self) -> int:
        return int(self.sample_rate * self.clip_duration)


CFG = TrainConfig()
DATA_DIR = PROJECT_ROOT / "data" / "birdclef-2026"
TRAIN_AUDIO_DIR = DATA_DIR / "train_audio"
OUTPUT_DIR = PROJECT_ROOT / "experiments" / "outputs" / "exp_002_train_audio_reproduction"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
amp_enabled = CFG.use_amp and device.type == "cuda"

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(CFG.seed)
torch.set_float32_matmul_precision("high")

{
    "device": str(device),
    "amp_enabled": amp_enabled,
    "output_dir": str(OUTPUT_DIR),
    **asdict(CFG),
}


In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
taxonomy_df = pd.read_csv(DATA_DIR / "taxonomy.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

species = sample_sub.columns[1:].tolist()
label_to_idx = {label: idx for idx, label in enumerate(species)}

train_df["primary_label"] = train_df["primary_label"].astype(str)
train_df["file_path"] = train_df["filename"].map(lambda name: TRAIN_AUDIO_DIR / name)
train_df = train_df[train_df["file_path"].map(Path.exists)].copy().reset_index(drop=True)

if CFG.max_rows is not None:
    train_df = train_df.sample(CFG.max_rows, random_state=CFG.seed).reset_index(drop=True)

if CFG.max_samples_per_class is not None:
    train_df = (
        train_df.groupby("primary_label", group_keys=False)
        .apply(lambda frame: frame.sample(min(len(frame), CFG.max_samples_per_class), random_state=CFG.seed))
        .reset_index(drop=True)
    )

class_counts = train_df["primary_label"].value_counts()
dataset_summary = {
    "rows": int(len(train_df)),
    "unique_primary_labels": int(train_df["primary_label"].nunique()),
    "taxonomy_classes": int(len(species)),
    "min_count": int(class_counts.min()),
    "median_count": float(class_counts.median()),
    "max_count": int(class_counts.max()),
    "rows_with_secondary_labels": int(train_df["secondary_labels"].fillna("[]").ne("[]").sum()),
}
display(pd.Series(dataset_summary, name="value").to_frame())
train_df.head(3)


In [ ]:
def build_holdout_split(df: pd.DataFrame, valid_frac: float, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    parts = []

    for label, group in df.groupby("primary_label", sort=False):
        indices = group.index.to_numpy(copy=True)
        rng.shuffle(indices)

        if len(indices) == 1:
            valid_size = 0
        else:
            valid_size = max(1, int(round(len(indices) * valid_frac)))
            valid_size = min(valid_size, len(indices) - 1)

        valid_idx = set(indices[:valid_size])
        labeled_group = group.copy()
        labeled_group["split"] = ["valid" if idx in valid_idx else "train" for idx in group.index]
        parts.append(labeled_group)

    return pd.concat(parts, axis=0).reset_index(drop=True)


split_df = build_holdout_split(train_df, CFG.valid_frac, CFG.seed)
train_split = split_df[split_df["split"] == "train"].reset_index(drop=True)
valid_split = split_df[split_df["split"] == "valid"].reset_index(drop=True)

split_summary = {
    "train_rows": int(len(train_split)),
    "valid_rows": int(len(valid_split)),
    "train_classes": int(train_split["primary_label"].nunique()),
    "valid_classes": int(valid_split["primary_label"].nunique()),
    "classes_missing_from_valid": int(train_split["primary_label"].nunique() - valid_split["primary_label"].nunique()),
}

split_df.to_csv(OUTPUT_DIR / "train_valid_manifest.csv", index=False)
display(pd.Series(split_summary, name="value").to_frame())
valid_split["primary_label"].value_counts().head(10)


## Data Pipeline

- Each sample is converted to a fixed 5-second waveform.
- Train samples use a random crop when the recording is longer than 5 seconds.
- Validation samples use a deterministic center crop.
- The initial reproduction baseline uses primary labels only by default.


In [ ]:
def parse_secondary_labels(value: str) -> list[str]:
    if not isinstance(value, str) or value in ["[]", ""]:
        return []
    cleaned = value.strip("[]").replace("'", "").replace('"', "")
    return [item.strip() for item in cleaned.split(",") if item.strip()]


class TrainAudioDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, cfg: TrainConfig, label_to_idx: dict[str, int], is_train: bool):
        self.frame = frame.reset_index(drop=True)
        self.cfg = cfg
        self.label_to_idx = label_to_idx
        self.is_train = is_train

    def __len__(self) -> int:
        return len(self.frame)

    def _crop_or_pad(self, audio: np.ndarray) -> np.ndarray:
        target_len = self.cfg.clip_samples
        if len(audio) >= target_len:
            if self.is_train:
                max_start = len(audio) - target_len
                start = 0 if max_start <= 0 else np.random.randint(0, max_start + 1)
            else:
                start = max(0, (len(audio) - target_len) // 2)
            audio = audio[start : start + target_len]
        else:
            audio = np.pad(audio, (0, target_len - len(audio)))
        return audio.astype(np.float32)

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        audio, _ = librosa.load(row["file_path"], sr=self.cfg.sample_rate, mono=True)
        audio = np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0)
        audio = self._crop_or_pad(audio)

        target = np.zeros(len(self.label_to_idx), dtype=np.float32)
        primary_label = row["primary_label"]
        if primary_label in self.label_to_idx:
            target[self.label_to_idx[primary_label]] = 1.0

        if self.cfg.use_secondary_labels:
            for secondary_label in parse_secondary_labels(row["secondary_labels"]):
                if secondary_label in self.label_to_idx:
                    target[self.label_to_idx[secondary_label]] = 1.0

        return {
            "audio": torch.from_numpy(audio),
            "target": torch.from_numpy(target),
            "primary_label": primary_label,
            "file_path": str(row["file_path"]),
        }


## Model And Metrics

This experiment intentionally reuses the same reference mel frontend and architecture that powered the first public baseline, but now trains from scratch on repository data.


In [ ]:
model_cfg = ReferenceModelConfig(num_classes=len(species))
SEDModel, MelTransform = build_reference_model_components(model_cfg)


class WaveformSEDClassifier(nn.Module):
    def __init__(self, model_cfg: ReferenceModelConfig):
        super().__init__()
        SEDModelCls, MelTransformCls = build_reference_model_components(model_cfg)
        self.mel = MelTransformCls()
        self.model = SEDModelCls()

    def forward(self, waveforms: torch.Tensor) -> dict[str, torch.Tensor]:
        mel = self.mel(waveforms)
        return self.model(mel)


def summarize_macro_auc(y_true: np.ndarray, y_pred: np.ndarray, species: list[str]) -> dict:
    metrics = score_macro_auc(y_true, y_pred, species)
    return {
        "macro_auc": metrics["macro_auc"],
        "scored_classes": metrics["scored_classes"],
        "skipped_no_positive": metrics["skipped_no_positive"],
        "skipped_no_negative": metrics["skipped_no_negative"],
    }


In [ ]:
train_dataset = TrainAudioDataset(train_split, CFG, label_to_idx, is_train=True)
valid_dataset = TrainAudioDataset(valid_split, CFG, label_to_idx, is_train=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    pin_memory=device.type == "cuda",
    drop_last=False,
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=device.type == "cuda",
    drop_last=False,
)

model = WaveformSEDClassifier(model_cfg).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.learning_rate, weight_decay=CFG.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, CFG.epochs))
scaler_device_type = "cuda" if device.type == "cuda" else "cpu"
scaler = torch.amp.GradScaler(scaler_device_type, enabled=amp_enabled)

{
    "train_batches": len(train_loader),
    "valid_batches": len(valid_loader),
    "parameters_millions": round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
}


In [ ]:
def run_train_epoch(model, loader, optimizer, criterion, scaler, device, amp_enabled: bool) -> float:
    model.train()
    loss_meter = 0.0
    sample_count = 0

    for batch in tqdm(loader, desc="train", leave=False):
        audio = batch["audio"].to(device, non_blocking=True)
        target = batch["target"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with (torch.amp.autocast(device_type=device.type, enabled=amp_enabled) if amp_enabled else nullcontext()):
            output = model(audio)
            loss = criterion(output["clipwise_logit"], target)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = audio.size(0)
        loss_meter += loss.item() * batch_size
        sample_count += batch_size

    return loss_meter / max(1, sample_count)


@torch.no_grad()
def run_valid_epoch(model, loader, criterion, device, amp_enabled: bool, species: list[str]) -> dict:
    model.eval()
    loss_meter = 0.0
    sample_count = 0
    all_targets = []
    all_probs = []

    for batch in tqdm(loader, desc="valid", leave=False):
        audio = batch["audio"].to(device, non_blocking=True)
        target = batch["target"].to(device, non_blocking=True)

        with (torch.amp.autocast(device_type=device.type, enabled=amp_enabled) if amp_enabled else nullcontext()):
            output = model(audio)
            loss = criterion(output["clipwise_logit"], target)

        probs = torch.sigmoid(output["clipwise_logit"]).detach().cpu().numpy()
        all_probs.append(probs)
        all_targets.append(target.detach().cpu().numpy())

        batch_size = audio.size(0)
        loss_meter += loss.item() * batch_size
        sample_count += batch_size

    y_true = np.concatenate(all_targets, axis=0)
    y_pred = np.concatenate(all_probs, axis=0)
    metrics = summarize_macro_auc(y_true, y_pred, species)
    metrics["valid_loss"] = loss_meter / max(1, sample_count)
    return metrics


def save_checkpoint(path: Path, model, optimizer, scheduler, scaler, epoch: int, metrics: dict, cfg: TrainConfig) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "epoch": epoch,
        "metrics": metrics,
        "config": asdict(cfg),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
    }
    torch.save(payload, path)


def move_optimizer_state(optimizer, device: torch.device) -> None:
    for state in optimizer.state.values():
        for key, value in state.items():
            if isinstance(value, torch.Tensor):
                state[key] = value.to(device)


def load_checkpoint(path: Path, model, optimizer, scheduler, scaler, device: torch.device) -> dict:
    checkpoint = torch.load(path, map_location="cpu")
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    move_optimizer_state(optimizer, device)
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    scaler_state = checkpoint.get("scaler_state_dict")
    if scaler_state:
        scaler.load_state_dict(scaler_state)
    return checkpoint


## Training Run

Set `RUN_TRAINING = True` when you are ready to launch the first reproduction run in a Torch-enabled environment.

- Default `num_workers=0` is intentional for notebook stability on macOS/Jupyter.
- The training cell can resume from `last_model.pt` and will fall back to `best_model.pt` when needed.
- A fresh `last_model.pt` checkpoint is now written after every completed epoch, so future interruptions are cheaper.
- If you later move `TrainAudioDataset` into a reusable module under `src/`, you can try increasing `num_workers` again.


In [ ]:
RUN_TRAINING = True
RESUME_TRAINING = True

history_path = OUTPUT_DIR / "history.csv"
best_checkpoint_path = OUTPUT_DIR / "best_model.pt"
last_checkpoint_path = OUTPUT_DIR / "last_model.pt"

history: list[dict] = []
best_macro_auc = -math.inf
start_epoch = 1
resume_path = None

if history_path.exists():
    history = pd.read_csv(history_path).to_dict("records")
    existing_macro = [float(row["macro_auc"]) for row in history if pd.notna(row.get("macro_auc"))]
    if existing_macro:
        best_macro_auc = max(existing_macro)

if RESUME_TRAINING:
    if last_checkpoint_path.exists():
        resume_path = last_checkpoint_path
    elif best_checkpoint_path.exists():
        resume_path = best_checkpoint_path

if resume_path is not None:
    checkpoint = load_checkpoint(resume_path, model, optimizer, scheduler, scaler, device)
    resume_epoch = int(checkpoint["epoch"])
    start_epoch = resume_epoch + 1
    checkpoint_macro_auc = float(checkpoint.get("metrics", {}).get("macro_auc", -math.inf))
    best_macro_auc = max(best_macro_auc, checkpoint_macro_auc)

    if history:
        history_df = pd.DataFrame(history)
        history = history_df[history_df["epoch"] <= resume_epoch].to_dict("records")

    print(
        {
            "resume_path": str(resume_path),
            "resume_epoch": resume_epoch,
            "start_epoch": start_epoch,
            "best_macro_auc": best_macro_auc,
            "device": str(device),
        }
    )

if RUN_TRAINING:
    if start_epoch > CFG.epochs:
        print(
            f"Checkpoint already reached epoch {start_epoch - 1}. "
            f"Increase CFG.epochs above {start_epoch - 1} to continue training."
        )
    else:
        for epoch in range(start_epoch, CFG.epochs + 1):
            train_loss = run_train_epoch(model, train_loader, optimizer, criterion, scaler, device, amp_enabled)
            valid_metrics = run_valid_epoch(model, valid_loader, criterion, device, amp_enabled, species)
            scheduler.step()

            row = {
                "epoch": epoch,
                "train_loss": train_loss,
                **valid_metrics,
                "learning_rate": optimizer.param_groups[0]["lr"],
            }
            history.append(row)
            history_df = pd.DataFrame(history)
            history_df.to_csv(history_path, index=False)

            print(row)

            save_checkpoint(last_checkpoint_path, model, optimizer, scheduler, scaler, epoch, row, CFG)

            if valid_metrics["macro_auc"] > best_macro_auc:
                best_macro_auc = valid_metrics["macro_auc"]
                save_checkpoint(best_checkpoint_path, model, optimizer, scheduler, scaler, epoch, row, CFG)

            if CFG.save_every_epoch:
                save_checkpoint(OUTPUT_DIR / f"epoch_{epoch:02d}.pt", model, optimizer, scheduler, scaler, epoch, row, CFG)
else:
    print("Training skipped. Set RUN_TRAINING = True to launch exp_002.")


In [ ]:
if history_path.exists():
    history_df = pd.read_csv(history_path)
    display(history_df)
else:
    print("No history.csv found yet.")


## Results And Notes

- Expected first outcome: a repository-native isolated-audio baseline with interpretable validation metrics.
- If this model trails the borrowed checkpoints badly, the gap may come from training recipe details rather than architecture alone.
- If it is close, soundscape finetuning becomes the main lever for improvement.


In [ ]:
result_snapshot = {
    "experiment_id": "exp_002",
    "status": "pending" if not history_path.exists() else "completed_or_in_progress",
    "config": asdict(CFG),
    "dataset_summary": dataset_summary,
    "split_summary": split_summary,
    "best_checkpoint_exists": best_checkpoint_path.exists(),
    "last_checkpoint_exists": last_checkpoint_path.exists(),
}

if history_path.exists():
    history_df = pd.read_csv(history_path)
    result_snapshot["best_valid_macro_auc"] = float(history_df["macro_auc"].max())
    result_snapshot["best_epoch"] = int(history_df.loc[history_df["macro_auc"].idxmax(), "epoch"])

with open(OUTPUT_DIR / "result_snapshot.json", "w", encoding="utf-8") as handle:
    json.dump(result_snapshot, handle, indent=2, sort_keys=True)

result_snapshot


## Next Steps

- Run the first full training pass.
- Compare the best validation macro ROC-AUC against the borrowed-checkpoint baseline.
- Promote the best native checkpoint into the next soundscape finetuning notebook.
